# Data Explorer — the whole app in a notebook

Every page of the app, in the same order and with the same layout logic (visuals first, then tables), as plain code with `cyfouri_data_explorer`. No UI: each cell shows one chart or one table.

## Contents
- 1. Overview
- 2. Columns
- 3. Variable
- 4. Compare
- 5. Correlations
- 6. Data
- 7. Time series
- 8. Machine learning
- 9. KPI
- 10. Synthetic data
- 11. Report


In [ ]:
from pathlib import Path

import cyfouri_data_explorer as dx

SAMPLES = Path("../../../samples")
traps = dx.open(SAMPLES / "typing_traps.csv")    # customer table full of typing traps
sales = dx.open(SAMPLES / "sales_timeseries.csv")  # daily sales by store
traps

## 1. Overview

The dataset's home page: a summary strip, the quality alerts, a toolbar (search, type filter, sort, Top N) and one card per used column.

### Summary strip

In [ ]:
import pandas as pd

ov = traps.overview()
pd.Series({
    "rows": ov.rows,
    "total rows": ov.total_rows,
    "columns": ov.columns,
    "used columns": ov.used_columns,
    "missing cells": ov.missing_cells,
    "missing %": ov.missing_pct,
    "duplicate rows": ov.duplicate_rows,
    "duplicate columns": len(ov.duplicate_columns),
    "size (bytes)": ov.size_bytes,
}).to_frame("value")

In [ ]:
pd.Series(ov.types, name="columns").to_frame()  # columns by type

### Alerts

In [ ]:
pd.DataFrame([dx.to_dict(alert) for alert in ov.alerts])[["severity", "message", "columns"]]

### Toolbar: search, type filter, sort

The cards can be filtered by name or type and sorted by file order, name, missing, distinct or |skewness|. Here: used columns sorted by missing share.

In [ ]:
types = dx.to_pandas(traps.types)[["name", "type", "excluded"]]
cards = types[~types.excluded].merge(dx.to_pandas(ov), left_on="name", right_on="column")
cards = cards.drop(columns=["column", "excluded"])
cards.sort_values("missing_pct", ascending=False).head(10)

In [ ]:
needle, type_filter = "i", "numeric_continuous"  # search box + type chip
cards[cards.name.str.contains(needle) & (cards.type == type_filter)].sort_values("skewness", key=abs, ascending=False)

### Variable cards

One card per column, drawn by its type (identifiers have no chart). A few representative cards:

In [ ]:
traps.plot("age")  # continuous: histogram

In [ ]:
traps.plot("age", "box")  # the card's box view

In [ ]:
traps.plot("nb_children")  # discrete: bars

In [ ]:
traps.plot("is_active")  # boolean

In [ ]:
traps.plot("is_active", "pie")  # the card's pie switch

In [ ]:
dx.plot(traps.chart("city", top_n=5), analysis_type="categorical_nominal")  # Top N = 5

In [ ]:
traps.plot("segment")  # ordinal category, in its order

In [ ]:
# A high-cardinality category gets no chart on its card, only a note and examples
zip_card = traps.chart("zip_code")
pd.Series({"kind": zip_card.kind, "distinct": zip_card.distinct, "message": zip_card.message, "examples": zip_card.examples}).to_frame("zip_code")

In [ ]:
traps.plot("signup_date")  # date: timeline

In [ ]:
traps.plot("last_login", "day_of_week")  # date: other views (month, day_of_week, hour)

In [ ]:
traps.plot("comment")  # text: length distribution

## 2. Columns

The type review page: every column with its detected type, role, confidence and reasons, a sparkline with examples, missing share and distinct count. Low confidence comes first.

In [ ]:
rank = {"low": 0, "medium": 1, "high": 2}
review = pd.DataFrame([
    {
        "column": c.name,
        "dtype": c.storage_dtype,
        "type": c.type,
        "role": c.role,
        "confidence": c.confidence,
        "why": " · ".join(c.reasons),
        "examples": ", ".join(map(str, (c.summary or {}).get("examples", []))),
        "missing %": round(100 * (c.summary or {}).get("null_count", 0) / max((c.summary or {}).get("rows", 1), 1), 1),
        "invalid": (c.summary or {}).get("invalid_count", 0),
        "distinct": (c.summary or {}).get("distinct_count"),
        "include": not c.excluded,
        "_rank": rank[c.confidence],
        "_pos": c.position,
    }
    for c in traps.types
])
review.sort_values(["_rank", "_pos"]).drop(columns=["_rank", "_pos"])

### Views: to review / excluded

In [ ]:
to_review = [c.name for c in traps.types if c.confidence != "high" and not c.overridden]
excluded = [c.name for c in traps.types if c.excluded]
pd.DataFrame({"view": ["all", "review", "excluded"], "count": [len(traps.types), len(to_review), len(excluded)], "columns": [None, to_review, excluded]})

### Date columns: range, step and derived parts

In [ ]:
pd.DataFrame([
    {"column": c.name, **{k: c.summary["date"][k] for k in ("min", "max", "step", "parts")}}
    for c in traps.types if c.type == "datetime" and c.summary and c.summary.get("date")
])

### Editing columns

Change a type, set an ordinal order, change a role, exclude columns (single or bulk). Same rules as the page; everything is undone right after so the next sections see the detected types.

In [ ]:
traps.retype("nb_children", "categorical_ordinal")
labels = [bar["label"] for bar in traps.column("nb_children").summary["histogram"]]
traps.set_order("nb_children", sorted(labels, reverse=True))  # Edit order
traps.set_role("income", "quantity")
traps.exclude("zip_code", "email")  # bulk exclude
edited = dx.to_pandas(traps.types).set_index("name")
edited.loc[["nb_children", "income", "zip_code", "email"], ["type", "detected_type", "role", "detected_role", "order", "overridden", "excluded"]]

In [ ]:
traps.plot("nb_children")  # the retyped column, drawn in its new order

In [ ]:
# Undo: back to the detected type, role and order; include again
traps.include("zip_code", "email")
traps.reset_types("nb_children", "income")
dx.to_pandas(traps.types).set_index("name").loc[["nb_children", "income", "zip_code", "email"], ["type", "role", "order", "overridden", "excluded"]]

## 3. Variable

The page of one column. A number gets metrics, its chart with views, then tabs: summary, tests, distribution, outliers (and categories for a discrete number). Other types get their chart and the panel of their type.

### Numeric column: `price_eur`

#### Metrics

In [ ]:
COLUMN = "price_eur"
profile = traps.variable(COLUMN)
d = profile.numeric.describe
ALPHA = 0.05
normal = all(test.p_value >= ALPHA for test in profile.numeric.normality.tests if test.p_value is not None)
pd.Series({
    "count": d.count,
    "missing": f"{d.missing} ({100 * d.missing / d.rows:.1f}%)",
    "mean": d.mean,
    "median": d.median,
    "std": d.std,
    "skewness": d.skewness,
    "normality": "normal" if normal else "not normal",
}).to_frame("value")

#### Chart and its views: histogram, KDE, ECDF, box, violin, QQ

In [ ]:
traps.plot(COLUMN, "histogram")

In [ ]:
traps.plot(COLUMN, "kde")

In [ ]:
traps.plot(COLUMN, "ecdf")

In [ ]:
traps.plot(COLUMN, "box")

In [ ]:
traps.plot(COLUMN, "violin")

In [ ]:
traps.plot(COLUMN, "qq")

#### Tab: Summary (counts, center and spread, quantiles, shape)

In [ ]:
GROUPS = {
    "counts": ["rows", "count", "missing", "infinite", "zeros", "negatives", "distinct"],
    "center & spread": ["mean", "median", "std", "cv", "iqr", "mad", "sum"],
    "quantiles": ["min", "p1", "p5", "p10", "q1", "median", "q3", "p90", "p95", "p99", "max"],
    "shape": ["skewness", "kurtosis"],
}
pd.DataFrame([(group, key, getattr(d, key)) for group, keys in GROUPS.items() for key in keys], columns=["group", "statistic", "value"]).set_index(["group", "statistic"])

#### Tab: Tests (normality at alpha = 0.05)

In [ ]:
norm = profile.numeric.normality
tests = pd.DataFrame([dx.to_dict(test) for test in norm.tests])
tests["normal"] = tests.p_value >= ALPHA
tests[["key", "statistic", "p_value", "normal", "n", "note"]]

#### Tab: Distribution (fitted laws ranked by AIC, QQ against the best law, transformations)

In [ ]:
fit = traps.fit(COLUMN)
traps.plot(COLUMN, "laws")  # best laws over the histogram

In [ ]:
dx.plot(fit, "qq")  # QQ against the best law (law="gamma" for another)

In [ ]:
dx.to_pandas(fit)[["label", "discrete", "aic", "bic", "ks"]]

In [ ]:
pd.DataFrame(dx.to_dict(fit)["transforms"])

#### Tab: Outliers (rule of each method, then the rows of the chosen method)

In [ ]:
outliers = traps.outliers(COLUMN, method="iqr")
dx.to_pandas(outliers)

In [ ]:
pd.DataFrame([dx.to_dict(row) for row in outliers.rows]).head(10)

#### Tab: Categories (a discrete number: `satisfaction`)

In [ ]:
cat = traps.variable("satisfaction").categorical
pd.Series({
    "top": f"{cat.top['label']} ({cat.top_share:.0%})",
    "distinct": cat.distinct,
    "missing": cat.missing,
    "entropy": cat.entropy,
    "evenness": cat.normalized_entropy,
    "herfindahl": cat.herfindahl,
    "rare": cat.rare_count,
    "imbalanced": cat.imbalanced,
    "uniform chi2 p": cat.chi2_p,
}).to_frame("value")

In [ ]:
full = traps.chart("satisfaction", top_n=1000)
bars = pd.DataFrame(full.bars)
bars.assign(share=bars["count"] / bars["count"].sum())

### Categorical column: `city`

Chart (every category), the pie view, the category metrics, then the full table.

In [ ]:
dx.plot(traps.chart("city", top_n=1000), analysis_type="categorical_nominal")

In [ ]:
traps.plot("city", "pie")

In [ ]:
cat = traps.variable("city").categorical
pd.Series({
    "top": f"{cat.top['label']} ({cat.top_share:.0%})",
    "distinct": cat.distinct,
    "missing": cat.missing,
    "entropy": cat.entropy,
    "evenness": cat.normalized_entropy,
    "herfindahl": cat.herfindahl,
    "rare": cat.rare_count,
    "imbalanced": cat.imbalanced,
    "uniform chi2 p": cat.chi2_p,
}).to_frame("value")

In [ ]:
bars = pd.DataFrame(traps.chart("city", top_n=1000).bars)
bars.assign(share=(bars["count"] / bars["count"].sum()).round(3))

### Date column: `signup_date` / `last_login`

Chart views (timeline, month, day of week, hour), then the date panel: metrics, gaps, calendar.

In [ ]:
traps.plot("signup_date", "timeline")

In [ ]:
traps.plot("signup_date", "month")

In [ ]:
traps.plot("signup_date", "day_of_week")

In [ ]:
traps.plot("last_login", "hour")

In [ ]:
date = traps.variable("signup_date").date
pd.Series({
    "range": f"{date.min[:10]} -> {date.max[:10]}",
    "granularity": date.granularity,
    "step": date.step,
    "periods": date.periods,
    "gaps": date.gap_count,
    "missing": date.missing,
    "has time": date.has_time,
}).to_frame("value")

In [ ]:
pd.DataFrame([dx.to_dict(gap) for gap in date.gaps]).head(10)  # missing periods (empty: no gap)

In [ ]:
traps.plot("signup_date", "calendar")

### Text column: `comment`

Length chart, text metrics, frequent words.

In [ ]:
traps.plot("comment")

In [ ]:
text = traps.variable("comment").text
pd.Series({
    "distinct": text.distinct,
    "missing": f"{text.missing} ({100 * text.missing / text.rows:.1f}%)",
    "lengths": f"{text.min_length} - {text.max_length} (mean {text.mean_length:.1f})",
}).to_frame("value")

In [ ]:
traps.plot("comment", "words")

### Identifier column: `email`

No chart, as in the app: uniqueness metrics, repeated values, value patterns.

In [ ]:
ident = traps.variable("email").identifier
pd.Series({
    "unique": f"{ident.unique_pct:.0f}%",
    "distinct": ident.distinct,
    "repeated values": ident.duplicated_values,
    "missing": ident.missing,
    "length": f"{ident.min_length} - {ident.max_length}",
}).to_frame("value")

In [ ]:
pd.DataFrame([dx.to_dict(x) for x in ident.duplicates], columns=["label", "count"])  # repeated values (empty: all unique)

In [ ]:
pd.DataFrame([dx.to_dict(x) for x in ident.patterns]).head(10)  # value patterns

## 4. Compare

Pick X and Y (optionally Z and a colour): the chart and the tests follow the types. Each pair below shows the chart first, then the tests panel as a table (recommended test first).

In [ ]:
import pandas as pd


def tests_table(comparison):
    """The tests panel: recommended first, effect size and magnitude flattened."""
    table = dx.to_pandas(comparison)
    if table.empty:
        return pd.DataFrame({"message": [comparison.message]})
    effect = table.pop("effect").apply(lambda e: e or {})
    table["effect"] = effect.apply(lambda e: e.get("name"))
    table["effect_value"] = effect.apply(lambda e: e.get("value"))
    table["magnitude"] = effect.apply(lambda e: e.get("magnitude"))
    table = table.sort_values("recommended", ascending=False, kind="stable")
    return table.round(4).reset_index(drop=True)


num_num = traps.compare("age", "income")
f"{num_num.kind}: {num_num.rows} of {num_num.total} rows used"

### Number × number

Scatter with the regression line (its R² is in the legend), the same points without it, and the density view.

In [ ]:
dx.plot(num_num, "points")

In [ ]:
dx.plot(num_num, "points", regression=False)

In [ ]:
dx.plot(num_num, "density")

In [ ]:
tests_table(num_num)

### Colour by a category

The colour picker accepts a category for number × number, number × category and date × number.

In [ ]:
dx.plot(traps.compare("age", "income", color="segment"), "points")

### Third number: 3D

With two numbers, a Z number turns the scatter into a 3D cloud (the default view).

In [ ]:
dx.plot(traps.compare("age", "income", "price_eur"))

### Whole numbers with few values

A whole number with at most 12 values reads as categories (`satisfaction` here); `numbers=[...]` keeps it as a number, like the page's *As numbers* link.

In [ ]:
dx.plot(traps.compare("satisfaction", "income"))

In [ ]:
dx.plot(traps.compare("satisfaction", "income", numbers=["satisfaction"]))

### Number × category

Box, violin and means (with confidence intervals), then the tests (Welch / Mann-Whitney / ANOVA / Kruskal, Levene).

In [ ]:
num_cat = traps.compare("income", "segment")
dx.plot(num_cat, "box")

In [ ]:
dx.plot(num_cat, "violin")

In [ ]:
dx.plot(num_cat, "means")

In [ ]:
tests_table(num_cat)

A second category in colour: the server puts the one with more values on the axis.

In [ ]:
num_cat_color = traps.compare("income", "segment", color="country")
print(num_cat_color.layout)
dx.plot(num_cat_color, "box")

### Category × category

Heatmap and stacked bars of the crosstab, the crosstab itself, then chi² / Fisher with Cramér's V.

In [ ]:
cat_cat = traps.compare("city", "segment")
dx.plot(cat_cat, "heatmap")

In [ ]:
dx.plot(cat_cat, "stacked")

In [ ]:
table = cat_cat.contingency
crosstab = pd.DataFrame(table.counts, index=table.rows, columns=table.columns)
crosstab.index.name, crosstab.columns.name = cat_cat.y, cat_cat.x
crosstab

In [ ]:
tests_table(cat_cat)

### Date × number

The aggregated series with its rolling mean and ± 1σ band; period and aggregate are chosen automatically, then the Mann-Kendall trend test with Sen's slope.

In [ ]:
date_num = traps.compare("signup_date", "income")
print(date_num.time)
dx.plot(date_num)

In [ ]:
tests_table(date_num)

In [ ]:
pd.DataFrame([dx.to_dict(date_num.trend)]).round(4)

The page's time options (period, aggregate, rolling window, alignment) and a colour, one series per value.

In [ ]:
from cyfouri_data_explorer.timeseries.series import Options as SeriesOptions

quarterly = SeriesOptions(period="quarter", aggregate="median", window=3, align="centered")
dx.plot(traps.compare("signup_date", "income", color="segment", options=quarterly))

In [ ]:
series = date_num.series[0]
pd.DataFrame({"period": series.periods, "value": series.values, "rolling_mean": series.rolling_mean, "rows": series.counts}).round(1)

## 5. Correlations

The mixed association matrix: r (or ρ) between numbers, η between a number and a category, Cramér's V between categories. The page shows the insights first, then the matrix, the focus on one variable and the ranked pairs.

In [ ]:
corr = traps.correlations()  # method="pearson" by default
pd.DataFrame([{"severity": i.severity, "kind": i.kind, "evidence": i.evidence, "insight": i.text} for i in corr.insights])

In [ ]:
dx.plot(corr)  # clustered order, as the page by default

In [ ]:
dx.plot(corr, clustered=False)  # file order

In [ ]:
dx.to_pandas(corr).round(2)

Columns left out of the matrix, and why.

In [ ]:
pd.DataFrame([dx.to_dict(e) for e in corr.matrix.excluded])

### Focus on one variable

Every association of `income`, strongest evidence first.

In [ ]:
focus = "income"
pairs = pd.DataFrame([dx.to_dict(p) for p in corr.matrix.pairs])
mine = pairs[(pairs.a == focus) | (pairs.b == focus)].copy()
mine["with"] = mine.a.where(mine.b == focus, mine.b)
rank = {"strong": 3, "moderate": 2, "weak": 1, "none": 0}
mine = mine.assign(_r=mine.evidence.map(rank), _v=mine.value.abs()).sort_values(["_r", "_v"], ascending=False)
mine[["with", "metric", "value", "evidence", "n"]].round(3).reset_index(drop=True)

### Strongest pairs

Every computed pair, strongest first, with the p-value adjusted over all pairs (Benjamini-Hochberg).

In [ ]:
pairs[["a", "b", "metric", "value", "q_value", "evidence", "n"]].rename(columns={"q_value": "adj. p"}).round(4).head(20)

### Method and columns

Spearman instead of Pearson, on a chosen set of columns.

In [ ]:
spearman = traps.correlations(method="spearman", columns=["age", "income", "price_eur", "satisfaction", "segment", "city"])
dx.plot(spearman)

## 6. Data

The rows, typed: first rows, a sample, duplicates or rows with missing values; a search over every column; a sort. Filters come from `where()`, which returns a filtered copy and leaves `traps` alone.

In [ ]:
dx.to_pandas(traps.rows(limit=20))

In [ ]:
dx.to_pandas(traps.rows(limit=20, mode="sample"))

In [ ]:
dupes = traps.rows(limit=20, mode="duplicates")
print(dupes.total, "duplicated rows")
dx.to_pandas(dupes)

In [ ]:
missing = traps.rows(limit=20, mode="missing")
print(missing.total, "rows with a missing value")
dx.to_pandas(missing)

In [ ]:
found = traps.rows(limit=20, search="paris", sort="income", desc=True)
print(found.total, "rows match 'paris'")
dx.to_pandas(found)

### Filters

The filter bar's operators. Conditions add up (AND); the values of `in` combine with OR. Here: three countries (FR, ES, DE), ages 25 to 60, a positive income, a known signup date not on a weekend.

In [ ]:
filtered = (
    traps.where("country", "in", ["FR", "ES", "DE"])
    .where("age", "between", (25, 60))
    .where("income", "gt", 0)
    .where("signup_date", "is_null", negate=True)
    .where("signup_date", "part_in", [6, 7], part="day_of_week", negate=True)
)
print(f"{len(filtered)} of {len(traps)} rows")
dx.to_pandas(filtered.rows(limit=20))

Text operators, and the row operators: one row per value (`unique`, first or last), or only the values seen twice or more (`duplicated`). A `dx.Condition` can also be built by hand.

In [ ]:
dx.to_pandas(traps.where("comment", "contains", "delivery").where("city", "starts_with", "P").rows(limit=20))

In [ ]:
repeated = traps.where(dx.Condition(column="city", op="duplicated", columns=["segment", "age"]))
print(len(repeated), "rows share city, segment and age with another row")
dx.to_pandas(repeated.rows(limit=20))

In [ ]:
one_per_city = traps.where(dx.Condition(column="city", op="unique", keep="last", order_by="signup_date"))
print(len(one_per_city), "rows, one per city")
dx.to_pandas(one_per_city.rows(limit=20))

Every analysis follows the filters: the same comparison on the filtered copy (and `traps` itself is unchanged).

In [ ]:
print(len(traps), "rows in traps,", len(filtered), "in the filtered copy")
dx.plot(filtered.compare("income", "segment"), "box")

### Export

The filtered rows to CSV, Parquet or Excel (the format follows the extension), all columns or a chosen few.

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp(prefix="dx_export_"))
written = [
    filtered.export(out / "filtered.csv"),
    filtered.export(out / "filtered.parquet"),
    filtered.export(out / "filtered.xlsx", columns=["customer_id", "country", "age", "income"]),
]
pd.DataFrame({"file": [p.name for p in written], "bytes": [p.stat().st_size for p in written]})

In [ ]:
pd.read_csv(out / "filtered.csv").head(10)

## 7. Time series

The page analyses one value over time. The bar at the top picks the time column, the value, the period, the aggregation, an optional group (and which of its series), and how to handle gaps; anything left unset is chosen by the app's rules. Then come the sections, in the page's order: overview, seasonality, decomposition, stationarity, autocorrelation, volatility, residuals, breaks, forecast.

### Setup

The choices actually used (`series.setup()`). `methods_table` turns a section's statistical tests into the same table the page shows under its chart.

In [ ]:
import dataclasses

import pandas as pd

series = sales.timeseries(time="date", value="revenue")  # period, aggregation, season: auto


def methods_table(result):
    """The tests of a section, as the page's table."""
    rows = pd.DataFrame([dataclasses.asdict(m) for m in result.methods])
    return rows.round(4).set_index("key")


setup = dataclasses.asdict(series.setup())
pd.Series({k: v for k, v in setup.items() if not isinstance(v, list)}, name="value").to_frame()

### Overview

The series, its rolling mean and ± 1σ band, then the key figures and the trend tests (Mann-Kendall, Sen's slope...).

In [ ]:
overview = series.overview()
dx.plot(overview)

In [ ]:
pd.DataFrame([dataclasses.asdict(overview.figures)])

In [ ]:
methods_table(overview)

#### Overview options

Period, aggregation, rolling window and its alignment (the rolling controls), gap filling, and which layers to draw.

In [ ]:
monthly = sales.timeseries(
    time="date", value="revenue", period="month", aggregate="mean",
    window=3, align="centered", gaps="interpolate",
)
dx.plot(monthly.overview(), layers=("values", "mean"))

#### Grouped series

With a group column, one series is analysed (`series=`) and the others can be drawn behind it (show all).

In [ ]:
paris = sales.timeseries(time="date", value="revenue", group="store", series="Paris")
dx.plot(paris.overview(), all_groups=True)

### Seasonality

One box per season (the strongest highlighted), the year-over-year overlay, then the seasonality tests.

In [ ]:
seasonality = series.seasonality()
dx.plot(seasonality, view=seasonality.profiles[0].key)

In [ ]:
dx.plot(seasonality, view="overlay")

In [ ]:
methods_table(seasonality)

### Decomposition

Trend, seasonal and remainder (`stl`, `mstl`, `classical_additive`, `classical_multiplicative`), with the strength of trend and seasonality.

In [ ]:
decomposition = series.decomposition("stl")
dx.plot(decomposition)

In [ ]:
parts = decomposition.result
pd.DataFrame({"method": [parts.method], "trend strength": [parts.strength_trend],
              "seasonal strength": [parts.strength_seasonal]}).round(3)

In [ ]:
dx.plot(series.decomposition("classical_multiplicative"))

### Stationarity

The series as tested (after the chosen transform), then ADF, KPSS, Phillips-Perron... The page suggests a transform; here it is applied.

In [ ]:
stationarity = series.stationarity()
dx.plot(stationarity)

In [ ]:
methods_table(stationarity)

In [ ]:
suggested = stationarity.suggestion.transform if stationarity.suggestion else "diff"
transformed = series.stationarity(transform=suggested)
dx.plot(transformed)

In [ ]:
methods_table(transformed)

### Autocorrelation

On the same transform as stationarity: the ACF and PACF correlograms, the fitted AR model (order by AIC, or your own), then Ljung-Box and Durbin-Watson.

In [ ]:
memory = series.autocorrelation(transform=suggested)
dx.plot(memory, view="acf")

In [ ]:
dx.plot(memory, view="pacf")

In [ ]:
pd.DataFrame([dataclasses.asdict(c) for c in memory.ar.coefficients]).round(4)

In [ ]:
pd.DataFrame([dataclasses.asdict(o) for o in memory.ar.orders]).round(1).set_index("order")

In [ ]:
methods_table(memory)

In [ ]:
# PACF method and AR order chosen by hand
manual = series.autocorrelation(transform=suggested, pacf="burg", ar_order=3)
pd.DataFrame([dataclasses.asdict(c) for c in manual.ar.coefficients]).round(4)

### Volatility

The residuals and their rolling spread, then the heteroscedasticity tests (Breusch-Pagan, White, ARCH-LM...). The spread window can be set.

In [ ]:
volatility = series.volatility()
dx.plot(volatility)

In [ ]:
methods_table(volatility)

In [ ]:
dx.plot(series.volatility(spread_window=8))

### Residuals

The distribution of the residuals (or of the series, or its returns): QQ plot, summary statistics, normality tests.

In [ ]:
residuals = series.residuals()
dx.plot(residuals)

In [ ]:
d = residuals.distribution.describe
pd.Series({k: getattr(d, k) for k in ("count", "mean", "std", "median", "iqr", "skewness", "kurtosis")},
          name="value").round(3).to_frame()

In [ ]:
pd.DataFrame([dataclasses.asdict(t) for t in residuals.distribution.normality.tests]).round(4)

In [ ]:
dx.plot(series.residuals(source="returns"))

### Breaks

Change points (search, cost, sensitivity) drawn on the series, the jumps, the break tests, then the anomalies (method and threshold).

In [ ]:
breaks = series.breaks()
dx.plot(breaks)

In [ ]:
pd.DataFrame([dataclasses.asdict(p) for p in breaks.points]).round(1)

In [ ]:
methods_table(breaks)

In [ ]:
pd.DataFrame([dataclasses.asdict(a) for a in breaks.anomalies]).drop(columns=["start", "end"]).round(2).head(10)

In [ ]:
tuned = series.breaks(search="binseg", cost="mean", sensitivity="high", anomaly="iqr", threshold=3.0)
dx.plot(tuned)

### Forecast

Models validated on rolling origins and ranked by MASE against the naive reference: the best model's forecast with its intervals, every model overlaid, then the scores. (Few models and a short horizon here, to stay fast.)

In [ ]:
forecast = series.forecast(horizon=8, models=["naive", "seasonal_naive", "holt", "theta"])  # ets_auto: ~20 s
dx.plot(forecast)

In [ ]:
dx.plot(forecast, overlay=True)

In [ ]:
scores = dx.to_pandas(forecast)
scores[[c for c in scores.columns if not isinstance(scores[c].iloc[0], (list, dict))]].round(3)

### Methods

Every section has a methods picker; ★ marks the ones run by default. Any list can be passed with `sales.timeseries(..., methods=[...])`.

In [ ]:
from cyfouri_data_explorer.timeseries import autocorrelation as ac, breaks as br, forecast as fc
from cyfouri_data_explorer.timeseries import methods as mt, stationarity as st, volatility as vo

catalog = {
    "trend": (mt.TREND, mt.DEFAULT_TREND), "seasonality": (mt.SEASONALITY, mt.DEFAULT_SEASONALITY),
    "stationarity": (st.STATIONARITY, st.DEFAULT_STATIONARITY),
    "autocorrelation": (ac.AUTOCORRELATION, ac.DEFAULT_AUTOCORRELATION),
    "volatility": (vo.VOLATILITY, vo.DEFAULT_VOLATILITY), "breaks": (br.BREAK_TESTS, br.DEFAULT_BREAK_TESTS),
    "forecast": (fc.MODELS, fc.DEFAULT_MODELS),
}
pd.DataFrame([{"section": s, "method": k, "default": "★" if k in default else ""}
              for s, (every, default) in catalog.items() for k in every])

## 8. Machine learning

The page asks one question first: *what do you want to predict?* Then the left column sets up the run (target, classes, model, settings, preparation, validation) and the right column shows the results of the run (verdict, scores, then the Performance / Features / Explain / Predictions tabs). Compare, Tune and Run compare open from the left column.

### 8.1 What do you want to predict?

Every column as a target, with its task (regression, binary, multiclass) and why it is or is not suggested.

In [ ]:
import numpy as np
import pandas as pd

dx.to_pandas(traps.ml_targets())

The columns of `typing_traps.csv` are drawn at random, so no model can beat chance on them. To show results that mean something, the rest of the section uses a small shop table where the outcome does depend on the inputs: `bought` (buy / skip) for classification, `spend` for regression.

In [ ]:
rng = np.random.default_rng(2)
n = 400
shop_frame = pd.DataFrame({
    "income": rng.normal(40_000, 9_000, n).round(0),
    "age": rng.integers(18, 75, n),
    "visits": rng.poisson(4, n),
    "city": rng.choice(["Paris", "Lyon", "Nice"], n),
})
shop_frame["bought"] = np.where(shop_frame["income"] + 1_500 * shop_frame["visits"] + rng.normal(0, 6_000, n) > 47_000, "buy", "skip")
shop_frame["spend"] = (shop_frame["income"] * 0.02 + 25 * shop_frame["visits"] + np.where(shop_frame["city"] == "Paris", 120, 0) + rng.normal(0, 60, n)).round(2)
shop = dx.from_pandas(shop_frame, "shop")
shop.to_pandas().head(8)

### 8.2 Classification: predict `bought`

#### Setup: target, task and validation

The validation (split, k-fold..., metric, threshold rule) is the line under the preparation in the left column; it is edited in the settings drawer.

In [ ]:
ml = shop.ml("bought", validation={"method": "split", "test_size": 0.2, "threshold": "f1", "seed": 42})
catalogue = ml.catalogue()
pd.DataFrame([
    {"setting": "target", "value": ml.target},
    {"setting": "task", "value": ml.task()},
    {"setting": "validation", "value": f"{ml.validation.method}, test {ml.validation.test_size:.0%}, seed {ml.validation.seed}"},
    {"setting": "metric", "value": ml.validation.metric or catalogue.default_metric},
    {"setting": "threshold rule", "value": ml.validation.threshold},
    {"setting": "metrics available", "value": ", ".join(catalogue.metrics)},
])

#### Model picker

The recommended models for this data first, then the whole catalogue by family.

In [ ]:
pd.DataFrame([{"model": r.key, "why": r.why} for r in catalogue.recommended])

In [ ]:
models = pd.DataFrame([
    {"key": m.key, "label": m.label, "family": m.family, "badges": ", ".join(m.badges), "installed": m.installed, "unavailable": m.unavailable or ""}
    for m in catalogue.models
])
models

#### Settings: the essential ones under the picker, all of them in the drawer

Shown for the random forest; the defaults are already adapted to this data.

In [ ]:
forest_view = next(m for m in catalogue.models if m.key == "random_forest")
pd.DataFrame([
    {"group": s.group, "setting": s.key, "label": s.label, "level": s.level, "default": s.default, "range": f"{s.minimum} to {s.maximum}" if s.minimum is not None else ", ".join(map(str, s.options)), "why": s.why}
    for s in forest_view.settings
]).sort_values(["level", "group"], key=lambda c: c.map({"essential": 0}).fillna(1) if c.name == "level" else c)

#### Preparation editor

Automatic preparation, column by column, and why (the *Columns* tab).

In [ ]:
preview = ml.preview("logistic_regression")
pd.DataFrame([
    {"column": d.column, "type": d.type, "use": d.use, "impute": d.impute, "transform": d.transform, "encoding": d.encoding, "scaling": d.scaling, "why": " ".join(d.why)}
    for d in preview.preparation.decisions
])

The *Preview* tab: the first rows once prepared, as the model sees them.

In [ ]:
pd.DataFrame(preview.rows, columns=preview.columns, index=pd.Index(preview.row_numbers, name="row")).head(8).round(3)

A custom preparation (*Columns*, *Missing values*, *Imbalance* and *Selection* tabs): drop `age`, impute numbers with the median, oversample the rarer class, keep features by mutual information.

In [ ]:
custom = {
    "columns": {"age": {"use": False}, "city": {"encoding": "onehot"}},
    "numeric_impute": "median",
    "imbalance": "oversample",
    "selection": [{"method": "mutual_info"}],
}
custom_preview = ml.preview("logistic_regression", preparation=custom)
pd.DataFrame([
    {"column": d.column, "use": d.use, "encoding": d.encoding, "scaling": d.scaling, "changed": ", ".join(d.manual)}
    for d in custom_preview.preparation.decisions
])

#### Estimate, then Train

In [ ]:
estimate = ml.estimate("logistic_regression")
pd.DataFrame([{"model": "logistic_regression", "seconds": round(estimate.seconds, 2), "fits": estimate.fits, "rows": estimate.rows}])

In [ ]:
model = ml.train("logistic_regression", settings={}, preparation=None)
classification = model.classification
pd.DataFrame([{
    "model": model.model, "task": model.task, model.primary: model.score,
    "accuracy": classification.accuracy, "majority baseline": classification.baseline_accuracy,
    "balanced accuracy": classification.balanced_accuracy, "threshold": classification.threshold,
    "train rows": model.train_rows, "test rows": model.test_rows, "seconds": round(model.seconds, 2),
}]).round(3)

All metrics (the expandable list under the score tiles).

In [ ]:
pd.Series(model.scores, name="value").rename_axis("metric").to_frame().round(4)

#### Results: Performance tab

Confusion matrix (counts, then % by row), the per-class table, then ROC, precision-recall and calibration.

In [ ]:
dx.plot(model, "confusion")

In [ ]:
dx.plot(model, "confusion_percent")

In [ ]:
pd.DataFrame([vars(r) for r in classification.report]).round(3)

In [ ]:
dx.plot(model, "roc")

In [ ]:
dx.plot(model, "pr")

In [ ]:
dx.plot(model, "calibration")

#### Results: Features tab

A linear model shows its top coefficients (odds ratios for a logistic regression), then the table with confidence intervals, p-values and VIF.

In [ ]:
dx.plot(model, "coefficients")

In [ ]:
pd.DataFrame([vars(c) for c in model.coefficients]).round(4)

#### Results: Explain tab

Permutation importance, the strongest pairs of inputs, SHAP values, then partial dependence.

In [ ]:
explanation = model.explain()
dx.plot(explanation, "permutation")

In [ ]:
dx.to_pandas(explanation).round(4)

In [ ]:
pd.DataFrame([vars(p) for p in explanation.pairs]).round(3)

In [ ]:
dx.plot(explanation, "shap")

In [ ]:
dx.plot(explanation, "dependence", column="income")

#### Results: Predictions tab

Every held-out row with its prediction and probability, then the explanation of one row (SHAP waterfall).

In [ ]:
pd.DataFrame([vars(p) for p in model.predictions]).head(12).round(3)

In [ ]:
dx.plot(explanation, "waterfall", row=explanation.shap.rows[0])

Predicting new rows with the trained pipeline.

In [ ]:
new_rows = pd.DataFrame({"income": [30_000, 45_000, 60_000], "age": [25, 40, 60], "visits": [1, 4, 8], "city": ["Nice", "Lyon", "Paris"], "spend": [630, 1000, 1520]})
pd.concat([new_rows, model.predict(new_rows).round(3)], axis=1)

A tree model on the same setup: a learning curve on the Performance tab, importances on the Features tab and the tree itself on the Explain tab.

In [ ]:
tree = ml.train("decision_tree", settings={"max_depth": 3})
dx.plot(tree, "importances")

In [ ]:
pd.DataFrame([vars(node) for node in tree.explain().tree]).head(15)

In [ ]:
boosted = ml.train("hist_gradient_boosting", settings={"max_iter": 60})
dx.plot(boosted, "learning_curve")

#### Compare models

Several models with the same validation, seed and preparation: best first, failures last. *Open* any of them to see its full results.

In [ ]:
compared = ml.compare(["logistic_regression", "random_forest", "knn", "gaussian_nb"], settings={"random_forest": {"n_estimators": 100}})
pd.DataFrame([
    {"model": m.model, m.primary if hasattr(m, "primary") else "score": getattr(m, "score", None), "seconds": round(getattr(m, "seconds", 0), 2), "error": getattr(m, "error", "")}
    for m in compared
]).round(4)

In [ ]:
dx.plot(compared)

#### Tune

Optuna on the training rows within a budget (here 6 trials or 15 s), with a sampler, a pruner and the search space (expert mode: ranges, log scale, fixed values).

In [ ]:
search = ml.tune(
    "random_forest", budget_seconds=15, trials=6, sampler="tpe", pruner="median",
    space=[
        {"key": "n_estimators", "kind": "int", "low": 50, "high": 150},
        {"key": "max_depth", "kind": "int", "low": 2, "high": 10},
        {"key": "min_samples_leaf", "kind": "int", "low": 1, "high": 10, "log": True},
    ],
)
dx.plot(search, "trials")

In [ ]:
pd.DataFrame([
    {"": "best", search.metric: search.best_value, **search.best_params},
    {"": "defaults", search.metric: search.default_value},
]).set_index("").round(4)

The importance of each setting (`dx.plot(search, "importance")`) needs at least 20 completed trials; with 6 trials the page shows every trial instead.

In [ ]:
pd.DataFrame([{"trial": t.number, "state": t.state, search.metric: t.value, "seconds": round(t.seconds, 2), **t.params} for t in search.trials]).set_index("trial").round(4)

*Use these settings*: train the model with the settings found.

In [ ]:
tuned = ml.train("random_forest", settings=search.best_params)
pd.DataFrame([{"run": "tuned random_forest", tuned.primary: tuned.score}, {"run": "logistic_regression", model.primary: model.score}]).round(4)

#### Run compare

Two or more runs of the history, same target and validation, on one chart.

In [ ]:
dx.plot([model, tuned])

### 8.3 Regression: predict `spend`

The same flow; the Performance tab shows predicted vs actual, the residual distribution and the residuals.

In [ ]:
reg = shop.ml("spend", validation={"method": "kfold", "folds": 5})
linear = reg.train("linear_regression")
regression = linear.regression
pd.DataFrame([{
    "model": linear.model, linear.primary: linear.score, "MAE": regression.mae, "baseline MAE": regression.baseline_mae,
    "RMSE": regression.rmse, "skill vs average": regression.skill, "validation": f"{reg.validation.method} ({reg.validation.folds} folds)",
}]).round(3)

In [ ]:
pd.DataFrame({"mean": linear.cross_validation.mean, "std": linear.cross_validation.std}).rename_axis("metric").round(4)

In [ ]:
dx.plot(linear, "predicted")

In [ ]:
dx.plot(linear, "residual_histogram")

In [ ]:
dx.plot(linear, "residuals")

In [ ]:
dx.plot(linear, "coefficients")

In [ ]:
pd.DataFrame([vars(c) for c in linear.coefficients]).round(4)

In [ ]:
dx.plot(linear.explain(), "permutation")

In [ ]:
pd.DataFrame([vars(p) for p in linear.predictions]).head(12).round(2)

In [ ]:
reg_compared = reg.compare(["linear_regression", "random_forest", "knn"], settings={"random_forest": {"n_estimators": 100}})
pd.DataFrame([{"model": m.model, m.primary: m.score, "seconds": round(m.seconds, 2)} for m in reg_compared]).round(4)

In [ ]:
dx.plot(reg_compared)

### 8.4 A number predicted as classes: `spend` cut into 3 classes

*Predict as: classes* on a numeric target: the classes panel cuts it (equal counts, equal width, natural breaks, custom limits or one class per value) and names the classes. Here: equal counts (quantiles).

In [ ]:
low, high = shop.to_pandas()["spend"].quantile([1 / 3, 2 / 3]).round(0)
classes = {"method": "quantile", "edges": [float(low), float(high)], "names": ["low", "medium", "high"]}
binned = shop.ml("spend", classes=classes)
shop_spend = shop.to_pandas()["spend"]
pd.DataFrame({
    "class": classes["names"],
    "range": [f"up to {low:g}", f"{low:g} to {high:g}", f"above {high:g}"],
    "rows": pd.cut(shop_spend, [-np.inf, low, high, np.inf], labels=classes["names"]).value_counts(sort=False).values,
    "task": binned.task(),
})

In [ ]:
binned_model = binned.train("logistic_regression")
dx.plot(binned_model, "confusion")

In [ ]:
pd.DataFrame([vars(r) for r in binned_model.classification.report]).round(3)

In [ ]:
dx.plot(binned_model, "roc")

## 9. KPI

The KPI page on `sales`: suggested KPIs, the tiles, the chart with its split suggestions, the Customize panel (chart type, show as, share) and the pivot table under it.

### Suggested KPIs

The chips above the builder, picked by rules from the column types, and the best columns to split by.

In [ ]:
import pandas as pd

suggested = sales.kpi_suggestions()
chips = pd.DataFrame(
    [{"rule": k.rule, "kpi": k.name, "definition": k.definition.model_dump(exclude_defaults=True)} for k in suggested.kpis]
    + [{"rule": "split", "kpi": f"+ {s}", "definition": None} for s in suggested.splits]
)
chips

### Tiles

The KPI "Total revenue" as the Builder computes it: last period, change against the previous one and against last year.

In [ ]:
revenue = next(k.definition for k in suggested.kpis if k.name == "Total revenue")
result = sales.kpi(revenue)
tiles = pd.DataFrame(dx.to_dict(result)["tiles"])
tiles[["label", "period", "value", "previous", "change", "last_year", "change_yoy"]]

### Chart

The rules choose the grain and the chart; the line under it says why.

In [ ]:
dx.plot(result)

In [ ]:
pd.DataFrame({"why": dx.to_dict(result)["resolved"]["why"]})

### Split by store

The "+ store" split suggestion: one series per store.

In [ ]:
by_store = sales.kpi({**revenue.model_dump(mode="json"), "groups": [{"column": "store"}]})
dx.plot(by_store)

### Customize: share view

The "% of total" chart drawn under the main one.

In [ ]:
dx.plot(by_store, "share")

### Customize: chart type and show as

The same result, other charts and other readings (value, share_total, change, change_pct, change_yoy, running).

In [ ]:
dx.plot(by_store, "line")

In [ ]:
dx.plot(result, mode="change_pct")

In [ ]:
dx.plot(by_store, "stacked_bars", mode="share_total")

In [ ]:
dx.plot(result, mode="running")

### Pivot table

Periods in rows, one column per store, with the total, as in the app's default layout.

In [ ]:
values = dx.to_pandas(by_store)
periods = list(dict.fromkeys(values["period"]))
pivot = values.pivot_table(index="period", columns="split", values="value", aggfunc="sum").reindex(periods)
pivot["Total"] = pivot.sum(axis=1)
pivot.round(0)

## 10. Synthetic data

The Synthetic data page on `traps`: the plan, a run, the report and its tabs (Columns, Sequences, Links, Checks, Privacy, Utility, Overview, Data), and two runs side by side.

### Plan

What each column becomes, and why.

In [ ]:
synth = traps.synthetic(use_gpu=False)
plan = synth.plan()
dx.to_pandas(plan)[["name", "type", "treatment", "rule", "reason"]]

In [ ]:
pd.DataFrame(dx.to_dict(plan)["methods"])[["key", "label", "why", "available", "badge"]]

### Generate

Two fast runs of 300 rows: sequential CART (the plan's choice) and a Gaussian copula.

In [ ]:
made = synth.generate("cart", rows=300)
copula = synth.generate("copula", rows=300)
report = dx.to_dict(made.report)
pd.DataFrame([report["header"]]).drop(columns=["types", "left_out"])

### Compare runs

The runs ticked in the history, scores side by side.

In [ ]:
dx.plot([made, copula])

In [ ]:
pd.DataFrame({"cart": dx.to_dict(made.report)["scores"], "copula": dx.to_dict(copula.report)["scores"]}).round(3)

### Report headline

Diagnostic, quality, detection and privacy scores, with their verdicts.

In [ ]:
dx.plot(made, "scores")

In [ ]:
pd.DataFrame({"score": report["scores"], "verdict": report["verdicts"]}).round(3)

### Columns tab

Every modelled column with its score, then one column real against synthetic.

In [ ]:
pd.DataFrame(report["columns"]).drop(columns=["real", "synthetic"])

In [ ]:
age = made.column("age")
dx.plot(age)

In [ ]:
dx.plot(age, "ecdf")

In [ ]:
pd.DataFrame(dx.to_dict(age)["stats"])

In [ ]:
segment = made.column("segment")
dx.plot(segment)

In [ ]:
pd.DataFrame(dx.to_dict(segment)["levels"])

### Sequences tab

Only for a TimeGAN run (the `deep` extra, slow on CPU): `dx.plot(run, "acf")` and `dx.plot(run, "samples")`. Not run here; the plan already shows the histories TimeGAN would learn.

In [ ]:
pd.DataFrame(dx.to_dict(plan)["sequence"]["candidates"])

### Links tab

The gap between real and synthetic associations, each side, the worst pairs, and one pair opened.

In [ ]:
dx.plot(made, "links")

In [ ]:
dx.plot(made, "links_real")

In [ ]:
dx.plot(made, "links_synthetic")

In [ ]:
pd.DataFrame(report["links"]["worst"])

In [ ]:
dx.plot(made.pair("age", "income"))

### Checks tab

Diagnostic checks, then quality: column shapes and pair trends.

In [ ]:
pd.DataFrame(report["diagnostic"]["checks"])

In [ ]:
pd.DataFrame(report["quality"]["shape_scores"])

In [ ]:
pd.DataFrame(report["quality"]["pair_scores"]).head(15)

### Privacy tab

Copies and distances to the closest real row, rare values, then the attribute-disclosure test (CAP).

In [ ]:
privacy = report["privacy"]
pd.Series({k: v for k, v in privacy.items() if not isinstance(v, (list, dict))}).to_frame("value")

In [ ]:
dx.plot(made, "dcr")

In [ ]:
pd.DataFrame(privacy["rare"] or [{"rare values": "No rare real value reproduced in the synthetic rows."}])

In [ ]:
cap = made.disclosure(known=["age", "city"], sensitive="segment")
pd.DataFrame(dx.to_dict(cap)["variants"])

### Utility tab

Train on synthetic, test on real (TSTR) against train on real (TRTR).

In [ ]:
found = made.utility("segment")
pd.DataFrame(dx.to_dict(found)["metrics"])

### Overview tab

Mean and spread of every numeric column, real against synthetic, and the PCA projection.

In [ ]:
dx.plot(made, "mean")

In [ ]:
dx.plot(made, "std")

In [ ]:
dx.plot(made, "pca")

### Data tab

The synthetic rows (`made.save(...)` writes them; `made.to_dataset()` explores them like any dataset).

In [ ]:
made.data.head(10)

## 11. Report

The app's HTML report, one self-contained file. In the app it is the Report button in the Overview page header, and also in the Data page's Export menu.

In [ ]:
import tempfile

out = Path(tempfile.mkdtemp()) / "typing_traps_report.html"
html = traps.report(out)
pd.DataFrame([{"file": str(out), "size_kb": round(out.stat().st_size / 1024, 1), "characters": len(html)}])